# Assignment Sesi 28 Tugas 2
Nama: Faraday Barr Fatahillah

Dataset = [h8_llm_genai](https://github.com/Sardiirfan27/h8_llm_genai/blob/main/data_mobil.csv)

**Tugas 2**

Buatlah aplikasi sederhana yang dapat melakukan pencarian brand mobil dan juga dapat memberikan informasi seputar spesifikasi dari setiap mobil. Misalkan dapat melakukan pencarian:
- Mobil yang cocok untuk keluarga
- Mobil Listrik●Mobil Autopilot
- Mobil hemat bahan bakar
- Mobil murah dibawah 250 juta dan keluaran tahun 2023

Gunakanlah “data_mobil.csvˮ sebagai informasi untuk melakukan pencarian tersebut. Download datasetnya di sini.Lakukanlah pencarian menggunakan Keyword Search (BM25 dan TF-IDF) dan Semantic Search dengan model selain dari model yang ada pada materi ini, misalkan Anda dapat menggunakan Gemini Embedding. Selain itu cobalah untuk simpan hasil embedding dengan Vector Database Pinecone dan ChromaDB.

In [1]:
# Import library
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from fastembed import TextEmbedding
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import chromadb
import time
from pinecone import Pinecone, ServerlessSpec

c:\Users\bobe\anaconda3\envs\bootcamp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
_df = pd.read_csv('data_mobil.csv')

_df['combined_text'] = (
    _df['brand'] + ' ' +
    _df['model'] + ' ' +
    _df['specs'] + ' ' +
    _df['description']
)

_df.head()

,id,brand,model,year,price_idr,specs,description,combined_text
0,1,Toyota,Avanza,2024,250000000,"1.5L, 4-cylinder, manual/automatic, 7-seater",Mobil keluarga 7 penumpang dengan konsumsi bah...,"Toyota Avanza 1.5L, 4-cylinder, manual/automat..."
1,2,Honda,Brio,2023,180000000,"1.2L engine, CVT, compact hatchback",City car kecil yang cocok untuk penggunaan har...,"Honda Brio 1.2L engine, CVT, compact hatchback..."
2,3,Mitsubishi,Xpander,2024,280000000,"1.5L engine, automatic, spacious interior",Mobil MPV dengan kabin luas dan nyaman untuk p...,"Mitsubishi Xpander 1.5L engine, automatic, spa..."
3,4,Suzuki,Ertiga,2023,240000000,"1.5L hybrid engine, manual/automatic",Mobil hybrid hemat bahan bakar dengan fitur mo...,"Suzuki Ertiga 1.5L hybrid engine, manual/autom..."
4,5,Toyota,Innova Zenix,2024,420000000,"2.0L hybrid, automatic, captain seat",MPV premium dengan teknologi hybrid dan kenyam...,"Toyota Innova Zenix 2.0L hybrid, automatic, ca..."


In [3]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(_df['combined_text'])


def tfidf_search(query, top_k=5):
    query_vector = vectorizer.transform([query])

    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()

    top_indices = similarities.argsort()[::-1][:top_k]

    results = _df.iloc[top_indices][[
        'brand',
        'model',
        'year',
        'price_idr',
        'description'
    ]]

    return results

In [4]:
query = 'mobil keluarga hemat bahan bakar'

results = tfidf_search(query)
print(results)

         brand    model  year  price_idr  \
3       Suzuki   Ertiga  2023  240000000   
0       Toyota   Avanza  2024  250000000   
5     Daihatsu     Ayla  2023  150000000   
18      Suzuki      XL7  2023  300000000   
2   Mitsubishi  Xpander  2024  280000000   

                                          description  
3   Mobil hybrid hemat bahan bakar dengan fitur mo...  
0   Mobil keluarga 7 penumpang dengan konsumsi bah...  
5   Mobil LCGC murah dan irit bahan bakar untuk pe...  
18  SUV hybrid dengan kapasitas besar dan efisiens...  
2   Mobil MPV dengan kabin luas dan nyaman untuk p...  


In [5]:
corpus = [doc.split(' ') for doc in _df['combined_text']]

bm25 = BM25Okapi(corpus)


def bm25_search(query, top_k=5):
    tokenized_query = query.split(' ')

    scores = bm25.get_scores(tokenized_query)

    top_indices = scores.argsort()[::-1][:top_k]

    results = _df.iloc[top_indices][[
        'brand',
        'model',
        'year',
        'price_idr',
        'description'
    ]]

    return results

In [6]:
query = 'mobil listrik modern'

results = bm25_search(query)
print(results)

      brand    model  year  price_idr  \
9   Hyundai  Ioniq 5  2024  750000000   
16    Tesla  Model 3  2024  900000000   
8    Wuling   Air EV  2024  300000000   
17  Hyundai    Creta  2023  350000000   
10      Kia    Sonet  2023  320000000   

                                          description  
9   Mobil listrik modern dengan teknologi canggih ...  
16  Mobil listrik dengan teknologi autopilot dan p...  
8   Mobil listrik kecil yang cocok untuk mobilitas...  
17  SUV modern dengan fitur keselamatan dan kenyam...  
10  SUV kompak dengan desain modern dan fitur lengkap  


In [7]:
model = TextEmbedding()

documents = _df['combined_text'].tolist()

embeddings = list(
    model.embed(documents)
)

embeddings = np.array(embeddings)

print(embeddings.shape)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
c:\Users\bobe\anaconda3\envs\bootcamp\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bobe\AppData\Local\Temp\fastembed_cache\models--qdrant--bge-small-en-v1.5-onnx-q. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.

(20, 384)


In [9]:
def semantic_search(query, top_k=5):

    # Embedding query
    query_embedding = list(
        model.embed([query])
    )[0]

    # Hitung similarity
    similarities = cosine_similarity(
        [query_embedding],
        embeddings
    ).flatten()

    # Ambil ranking tertinggi
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Ambil hasil
    results = _df.iloc[top_indices][[
        'brand',
        'model',
        'year',
        'price_idr',
        'description'
    ]]

    return results

In [10]:
query = 'mobil untuk keluarga besar dan nyaman'

results = semantic_search(query)
print(results)

         brand    model  year  price_idr  \
5     Daihatsu     Ayla  2023  150000000   
19    Daihatsu   Terios  2023  260000000   
2   Mitsubishi  Xpander  2024  280000000   
10         Kia    Sonet  2023  320000000   
8       Wuling   Air EV  2024  300000000   

                                          description  
5   Mobil LCGC murah dan irit bahan bakar untuk pe...  
19  SUV terjangkau dengan performa cukup untuk keb...  
2   Mobil MPV dengan kabin luas dan nyaman untuk p...  
10  SUV kompak dengan desain modern dan fitur lengkap  
8   Mobil listrik kecil yang cocok untuk mobilitas...  


In [11]:
client = chromadb.Client()

collection = client.create_collection(
    name='mobil_collection'
)

for i, row in _df.iterrows():

    collection.add(
        ids=[str(i)],
        documents=[row['combined_text']],
        embeddings=[embeddings[i].tolist()],
        metadatas=[{
            'brand': row['brand'],
            'model': row['model'],
            'year': int(row['year']),
            'price': int(row['price_idr'])
        }]
    )

In [12]:
query = 'mobil listrik murah'

query_embedding = list(
    model.embed([query])
)[0]

results = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=3
)

print(results)

{'ids': [['5', '9', '8']], 'embeddings': None, 'documents': [['Daihatsu Ayla 1.0L engine, manual, compact Mobil LCGC murah dan irit bahan bakar untuk penggunaan sehari-hari', 'Hyundai Ioniq 5 Electric, 58 kWh battery, fast charging Mobil listrik modern dengan teknologi canggih dan jarak tempuh jauh', 'Wuling Air EV Electric motor, battery 26 kWh Mobil listrik kecil yang cocok untuk mobilitas dalam kota']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'model': 'Ayla', 'year': 2023, 'price': 150000000, 'brand': 'Daihatsu'}, {'model': 'Ioniq 5', 'brand': 'Hyundai', 'year': 2024, 'price': 750000000}, {'price': 300000000, 'year': 2024, 'brand': 'Wuling', 'model': 'Air EV'}]], 'distances': [[0.756255030632019, 0.8151558637619019, 0.8415855169296265]]}


In [18]:
pc = Pinecone(api_key="pcsk_45vjW7_Sm8Mf9MDeamqFw4gjkPguJn1CtZH9oingfJmi9FGu6Y2moL3vGGFxUuYtZFoj2h")

index_name = "mobil-semantic-fastembed"

embedding_dimension = embeddings.shape[1]

existing_indexes = pc.list_indexes().names()

if index_name in existing_indexes:

    index_info = pc.describe_index(index_name)

    if index_info.dimension != embedding_dimension:

        print(
            f"Index exists with dimension "
            f"{index_info.dimension}, deleting and recreating..."
        )

        pc.delete_index(index_name)

        existing_indexes = []

if index_name not in existing_indexes:

    pc.create_index(
        name=index_name,
        dimension=embedding_dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

print("Pinecone index ready!")

Pinecone index ready!


In [19]:
vectors = []

for i, row in _df.iterrows():

    vectors.append({
        'id': str(i),
        'values': embeddings[i].tolist(),
        'metadata': {
            'brand': row['brand'],
            'model': row['model'],
            'description': row['description']
        }
    })

index.upsert(vectors=vectors)

UpsertResponse(upserted_count=20)

In [20]:
query = 'mobil keluarga modern'

query_embedding = list(
    model.embed([query])
)[0]

result = index.query(
    vector=query_embedding.tolist(),
    top_k=3,
    include_metadata=True
)

print(result)

QueryResponse(matches=[ScoredVector(id='5', score=0.674160063, values=[], metadata={'brand': 'Daihatsu', 'description': 'Mobil LCGC murah dan irit bahan bakar untuk penggunaan sehari-hari', 'model': 'Ayla'}), ScoredVector(id='3', score=0.666736662, values=[], metadata={'brand': 'Suzuki', 'description': 'Mobil hybrid hemat bahan bakar dengan fitur modern dan ramah lingkungan', 'model': 'Ertiga'}), ScoredVector(id='0', score=0.644550383, values=[], metadata={'brand': 'Toyota', 'description': 'Mobil keluarga 7 penumpang dengan konsumsi bahan bakar irit dan harga terjangkau', 'model': 'Avanza'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Mon, 18 May 2026 14:18:17 GMT', 'content-type': 'application/json', 'content-length': '603', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '1', 'x-pinecone-request-latency-ms': '48', 'x-envoy-upstream-service-time': '45', 'x-pinecone-response-duration-ms': '49', 'grpc-status': '

In [25]:
while True:
    print('=== Sistem Pencarian Mobil ===')
    print('1. TF-IDF')
    print('2. BM25')
    print('3. Semantic Search')
    print('4. Quit')

    choice = input('Pilih metode pencarian: ')
    
    if choice == '1':
        query = input('Masukkan pencarian mobil: ')
        hasil = tfidf_search(query)

    elif choice == '2':
        query = input('Masukkan pencarian mobil: ')
        hasil = bm25_search(query)

    elif choice == '3':
        query = input('Masukkan pencarian mobil: ')
        hasil = semantic_search(query)

    elif choice == '4':
        print('Terima kasih telah menggunakan sistem pencarian.')
        break

    else:
        print('Pilihan tidak valid')
        hasil = None

    if hasil is not None:
        print(f'\n\nPencarian: {query}', end='\n')
        print('\nHasil Pencarian:')
        print(hasil)
        print('\n' + '='*50 + '\n')

=== Sistem Pencarian Mobil ===
1. TF-IDF
2. BM25
3. Semantic Search
4. Quit


Pencarian: Mobil yang cocok untuk keluarga

Hasil Pencarian:
         brand    model  year  price_idr  \
8       Wuling   Air EV  2024  300000000   
1        Honda     Brio  2023  180000000   
2   Mitsubishi  Xpander  2024  280000000   
19    Daihatsu   Terios  2023  260000000   
7       Toyota   Feruza  2022  200000000   

                                          description  
8   Mobil listrik kecil yang cocok untuk mobilitas...  
1   City car kecil yang cocok untuk penggunaan har...  
2   Mobil MPV dengan kabin luas dan nyaman untuk p...  
19  SUV terjangkau dengan performa cukup untuk keb...  
7   SUV lama dengan harga terjangkau cocok untuk o...  


=== Sistem Pencarian Mobil ===
1. TF-IDF
2. BM25
3. Semantic Search
4. Quit


Pencarian: Mobil Listrik

Hasil Pencarian:
       brand    model  year  price_idr  \
16     Tesla  Model 3  2024  900000000   
9    Hyundai  Ioniq 5  2024  750000000   
8     Wulin